In [7]:
%pip install -q --no-cache-dir \
    transformers==4.53.3 \
    safetensors==0.5.3 \
    trl==0.19.1 \
    peft==0.17.1 \
    accelerate==1.8.1 \
    bitsandbytes==0.46.1 \
    datasets==3.6.0 \
    scikit-learn==1.7.2 \
    sentencepiece==0.2.0

In [8]:

import numpy as np
import pandas as pd
import torch
import os
import random
from trl import *
from peft import *
import zipfile
from pathlib import Path
from transformers import *
from scipy.sparse import hstack
from datasets import Dataset

seed = 42
model_id = "Qwen/Qwen3-4B-Instruct-2507"

MAX_LENGTH = 1024
MAX_PROMPT_LENGTH = 256
MAX_COMPLETION_LENGTH = 768

out = Path("/content/outputs")
sft_d = out / "sft"
dp_d = out / "dpo"
rwm_d = out / "rwd_m"
dpo_qd = out / "dpo_q"
simp_d = out / "simpo"

os.environ["PYTHONHASHSEED"] = str(seed)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
set_seed(seed)

out.mkdir(parents=True, exist_ok=True)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report

In [10]:
from pathlib import Path
import json
import zipfile

archive_path = Path(
    "/content/ml-olympiad-red-task-c1005bf0-8695-451a-9616-87c8687dce27.zip"
)

with zipfile.ZipFile(archive_path) as archive:
    archive.extractall("/content")

data_dir = Path("/content/ml-olympiad-red-task/data")


def read_jsonl(filename):
    with open(data_dir / filename, encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]


kid_adult = read_jsonl("kid_adult.jsonl")
good_bad = read_jsonl("good_bad.jsonl")
public_style = read_jsonl("public_test_style.jsonl")
public_quality = read_jsonl("public_test_quality.jsonl")
